# Mini-progetto 2 — Analizzatore di misurazioni

Questo esempio usa una lista di dizionari per rappresentare record simili alle righe di un dataset. La pipeline pulisce i dati, raggruppa per stazione e produce statistiche sulla temperatura.

## Dati di partenza

In [1]:
measuraments = [
    {"station": "A", "temperature": 18.5, "valid": True},
    {"station": "B", "temperature": 21.0, "valid": True},
    {"station": "A", "temperature": None, "valid": False},
    {"station": "A", "temperature": 19.2, "valid": True},
    {"station": "B", "temperature": 20.6, "valid": True},
    {"station": "C", "temperature": "error", "valid": True},
]

## Implementazione completa

In [ ]:
from __future__ import annotations

Record = dict[str, object]
Grouped = dict[str, list[float]]

def parse_temperature(value: object) -> float:
    if value is None:
        raise ValueError("Temperatura mancante.")
    
    try:
        temperature = float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"Temperatura non numerica: {value!r}") from error
    
    if not -80.0 <= temperature <= 60.0:
        raise ValueError("Temperatura fuorii intervallo: {temperature}")

    return temperature

def clean_measurements(records: list[Record]) -> tuple[list[Record], list[str]]:
    cleaned = []
    errors = []

    for index, record in enumerate(records):
        if not record.get("valid", False):
            errors.append(f"Riga {index}: record marcato come non valido.")
            continue

        station = str(record.get("station", "")).strip()
        if not station:
            errors.append(f"Riga {index}: stazione mancante")
            continue

        try:
            temperature = parse_temperature(record.get("temperature"))
        except ValueError as error:
            errors.append(f"Riga {index}: {error}")
            continue

        cleaned.append({
            "station": station,
            "temperature": temperature,
        })

    return cleaned, errors

def group_temperatures(records: list[Record]) -> Grouped:
    grouped: Grouped = {}

    for record in records:
        station = str(record["station"])
        temperature = float(record["temperature"])
        grouped.setdefault(station, []).append(temperature)

    return grouped

def summarize_groups(grouped: Grouped) -> dict[str, dict[str, float]]:
    return {
        station: {
            "mean": sum(values) / len(values),
            "minimum": min(values),
            "maximum": max(values),
        }
        for station, values in grouped.items()
        if values
    }

def main() -> None:
    measuraments = [
        {"station": "A", "temperature": 18.5, "valid": True},
        {"station": "B", "temperature": 21.0, "valid": True},
        {"station": "A", "temperature": None, "valid": False},
        {"station": "A", "temperature": 19.2, "valid": True},
        {"station": "B", "temperature": 20.6, "valid": True},
        {"station": "C", "temperature": "error", "valid": True},
    ]

    cleaned, errors = clean_measurements(measuraments)
    grouped = group_temperatures(cleaned)
    summaries = summarize_groups(grouped)

    print("Record puliti:", cleaned)
    print("Errori:")
    for error in errors:
        print("-", error)

    print("Statistiche:")
    for station, stats in summaries.items():
        print(station, stats)

if __name__ == "__main__":
    main()
